# RAG capstone — one realistic app, end to end

*The `gen_ai/` finale. It reuses every earlier notebook; the traditional-ML analog is `ml/j_capstone_end_to_end`.*

## What this is

`b_tracing_a_multistep_app` faked RAG with a toy lexical retriever so the *tracing* lesson stayed simple. This capstone does it **for real** — a proper vector database and embeddings — and threads the whole track on one app, exactly as `ml/j_capstone_end_to_end` threads the ML spine:

1. **Ingest & index** a document corpus into a **Milvus** vector store (real embeddings).
2. **Build the RAG** with **LlamaIndex**, traced by one-line `mlflow.llama_index.autolog()` (← `a_`/`b_`/`d_`).
3. **Evaluate** retrieval *and* answer quality with the LLM-judge (← `c_`).
4. **Govern the prompt** via the registry (← `e_`), and collect **feedback + monitor** live traffic (← `g_`).
5. **Serve** it the models-from-code way (← `f_`).

Each step *reuses* an earlier notebook's concept rather than re-teaching it.

## Prerequisites

- **Tracking server on 5001.**
- **Azure** for embeddings (`text-embedding-3-small`), the generation LLM, and the judge — credentials in `.env`. This capstone is deliberately **Azure-powered** for a realistic, reliable pipeline; the local swaps (Ollama embeddings `nomic-embed-text` + an Ollama LLM) are noted but slower.
- **Deps** (added): `llama-index-core`, `llama-index-vector-stores-milvus`, `llama-index-embeddings-azure-openai`, `llama-index-llms-azure-openai`, `pymilvus` (bundles **Milvus Lite** — an embedded Milvus, no Docker).

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

ENDPOINT = os.environ["AZURE_OPENAI_BASE_URL"].removesuffix("/openai").rstrip("/")
KEY      = os.environ["AZURE_OPENAI_API_KEY"]
APIVER   = os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21")

# judge config (same mapping as c_genai_evaluation)
os.environ["AZURE_API_KEY"], os.environ["AZURE_API_BASE"], os.environ["AZURE_API_VERSION"] = KEY, ENDPOINT, APIVER
JUDGE = f"azure:/{os.environ.get('AZURE_OPENAI_LIGHT_MODEL', 'gpt-5.4-nano')}"

from llama_index.core import Settings
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding
from llama_index.llms.azure_openai import AzureOpenAI

Settings.embed_model = AzureOpenAIEmbedding(
    deployment_name=os.environ.get("AZURE_OPENAI_EMBED_MODEL", "deepwiki-embeddings"),
    model="text-embedding-3-small", azure_endpoint=ENDPOINT, api_key=KEY, api_version=APIVER)
Settings.llm = AzureOpenAI(
    engine=os.environ.get("AZURE_OPENAI_LIGHT_MODEL", "gpt-5.4-nano"),
    model=os.environ.get("AZURE_OPENAI_LIGHT_MODEL", "gpt-5.4-nano"),
    azure_endpoint=ENDPOINT, api_key=KEY, api_version=APIVER)

import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("genai-rag-capstone")
mlflow.llama_index.autolog()   # one line: trace the whole RAG

## Step 1a — the document corpus

A small, self-contained knowledge base about MLflow stands in for *your* documents. In a real app you'd point `SimpleDirectoryReader("./docs")` at a folder of files instead — the rest is identical.

In [ ]:
from llama_index.core import Document

FACTS = [
    "MLflow Tracking logs parameters, metrics, tags, and artifacts for every run, keyed by a run id.",
    "An MLflow Experiment groups related runs so you can compare them in the UI.",
    "mlflow.autolog() automatically logs params, metrics, and the model when you call fit().",
    "The MLflow Model Registry version-controls models and assigns aliases such as champion and challenger.",
    "Moving a model alias like @champion changes which version downstream serving loads.",
    "mlflow models serve starts a REST endpoint that answers POST requests at /invocations.",
    "mlflow models build-docker packages a logged model into a container image.",
    "MLflow Tracing records spans (inputs, outputs, latency) for each step of a GenAI app.",
    "mlflow.openai.autolog() traces calls made through the openai client, including to Ollama.",
    "mlflow.genai.evaluate() scores GenAI outputs with LLM-as-judge scorers like RelevanceToQuery.",
    "The MLflow Prompt Registry versions prompt templates and serves them by alias, e.g. @production.",
    "mlflow.dspy.autolog() records a DSPy optimization run, including the optimized program.",
    "mlflow.log_feedback attaches human or automated assessments to a trace.",
    "RetrievalGroundedness is a scorer that checks whether an answer is supported by the retrieved context.",
]
docs = [Document(text=t) for t in FACTS]
print(len(docs), "documents")

## Step 1b — embed & index into Milvus

`MilvusVectorStore` with a local `uri` uses **Milvus Lite** — a real Milvus engine (real ANN index) embedded in-process, persisted to `milvus_rag.db`. Each document is embedded by the Azure model and stored.

**Gotcha (version drift):** `milvus-lite 3.0`'s *search* with the default `output_fields=["*"]` doesn't return scalar fields, so the LlamaIndex connector gets empty text back. Passing a non-empty `output_fields` flips it to an explicit field list (and it appends the text field), which round-trips correctly. This is a real, current incompatibility — exactly the kind of drift this repo flags.

In [ ]:
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.milvus import MilvusVectorStore

vector_store = MilvusVectorStore(
    uri="milvus_rag.db",          # Milvus Lite: embedded, persisted to this file
    dim=1536,                      # text-embedding-3-small
    overwrite=True,
    output_fields=["doc_id"],    # workaround for the milvus-lite 3.0 search gotcha above
)
index = VectorStoreIndex.from_documents(
    docs, storage_context=StorageContext.from_defaults(vector_store=vector_store))
print("indexed into Milvus Lite")

## Step 2 — query the RAG (traced)

The query engine retrieves the top chunks, stuffs them into the prompt, and answers. `autolog` records the whole thing — and now the **`RETRIEVER` span is real** (real chunks, real scores), not the toy one from `b_`.

In [ ]:
query_engine = index.as_query_engine(similarity_top_k=3)

response = query_engine.query("How do I expose a trained model as a REST endpoint?")
print(response, "\n")
print("retrieved from:")
for n in response.source_nodes:
    print(f"  [{n.score:.3f}] {n.node.get_content()[:70]}")

Open <http://127.0.0.1:5001> → **Traces**: the RAG trace shows the retriever span (the chunks it pulled and their scores) feeding the LLM span. This is `b_`'s lesson on a production-shaped app.

## Step 3 — evaluate retrieval *and* answers (← `c_`)

`mlflow.genai.evaluate` runs a question set through the RAG and judges it. `RetrievalGroundedness` is the RAG-specific scorer that finally has a real retriever to grade: *is the answer supported by what was retrieved?* — alongside `RelevanceToQuery`.

In [ ]:
from mlflow.genai.scorers import RelevanceToQuery, RetrievalGroundedness

def rag(question: str) -> str:
    return str(query_engine.query(question))

questions = [
    {"inputs": {"question": "How do I serve a model over REST?"}},
    {"inputs": {"question": "What does the model registry do?"}},
    {"inputs": {"question": "How are GenAI outputs scored in MLflow?"}},
]
result = mlflow.genai.evaluate(
    data=questions, predict_fn=rag,
    scorers=[RelevanceToQuery(model=JUDGE), RetrievalGroundedness(model=JUDGE)])
result.metrics

## Step 4 — govern the RAG prompt (← `e_`)

The instruction that turns retrieved context into an answer is itself a **prompt** — so register it. We use LlamaIndex's variable names (`{{context_str}}` / `{{query_str}}`) so the registered prompt drops straight into the query engine; promoting a new version then reaches the app by moving the `@production` alias, no code change.

In [ ]:
from llama_index.core import PromptTemplate

template = ("You are an MLflow assistant. Answer the question using ONLY the context; "
            "if it isn't there, say you don't know.\n\n"
            "Context:\n{{context_str}}\n\nQuestion: {{query_str}}\nAnswer:")
mlflow.genai.register_prompt(name="rag-qa", template=template, commit_message="v1: grounded QA")
mlflow.genai.set_prompt_alias(name="rag-qa", alias="production", version=1)

prod = mlflow.genai.load_prompt("prompts:/rag-qa@production")
governed = index.as_query_engine(
    similarity_top_k=3,
    text_qa_template=PromptTemplate(prod.to_single_brace_format()))  # {{x}} -> {x} for LlamaIndex
print(governed.query("What is mlflow models serve?"))

## Step 5 — feedback & monitoring (← `g_`)

Attach a user rating to a real RAG trace, then monitor quality by re-scoring recent traces — the same loop as `g_`, now on the capstone app.

In [ ]:
from mlflow.entities import AssessmentSource

governed.query("How do I version a prompt in MLflow?")
mlflow.flush_trace_async_logging()                          # persist before attaching feedback
recent = mlflow.search_traces(max_results=10)

mlflow.log_feedback(
    trace_id=recent.iloc[0]["trace_id"], name="user_rating", value="thumbs_up",
    source=AssessmentSource(source_type="HUMAN", source_id="alice@example.com"),
    rationale="Grounded and correct.")

# monitoring: re-score recent production traffic with the judge
print("monitoring:", mlflow.genai.evaluate(
    data=recent, scorers=[RelevanceToQuery(model=JUDGE)]).metrics)

## Step 6 — serve it (← `f_`)

Serving is exactly `f_genai_app_serving`'s pattern applied to this app: a **models-from-code** script whose `load_context` rebuilds the embed model, reconnects to `milvus_rag.db`, and constructs the query engine, and whose `predict` runs `query_engine.query(...)` — loading `rag-qa@production` so prompt promotion ships with no redeploy. Then `mlflow.pyfunc.log_model(python_model=...)` and `mlflow models serve -p 5002`. See `f_` for the full serve + `/invocations` walkthrough; the only change is what the model script builds.

*(Not re-run here to keep the capstone focused — the serving mechanics are identical to `f_`.)*

## The GenAI track is complete

You've built a real RAG app and run it through the whole MLOps loop — **index → trace → evaluate → govern prompts → serve → feedback/monitor** — reusing `a_`–`h_` rather than re-teaching them. That's the GenAI parallel of `ml/j_capstone_end_to_end`, and the finale of the `gen_ai/` track.